In [ ]:
!pip install -q xgboost shap

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
    RandomizedSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)

from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries loaded.")

In [ ]:
def find_file(filename):
    candidates = [
        filename,
        f"/content/{filename}",
        f"/mnt/data/{filename}"
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"{filename} not found. Upload it to Colab.")

train = pd.read_csv(find_file("dataset_Repeat_training.csv"))
test = pd.read_csv(find_file("dataset_Repeat_testing.csv"))

TARGET = "Transported"
ID = "PassengerId"

print("train:", train.shape, "| test:", test.shape)
display(train.head())

In [ ]:
print("TRAIN INFO")
train.info()

print("\nTEST INFO")
test.info()

print("\nDuplicate rows in train:", train.duplicated().sum())
print("Duplicate rows in test :", test.duplicated().sum())
print("Duplicate PassengerIds in train:", train[ID].duplicated().sum())
print("Duplicate PassengerIds in test :", test[ID].duplicated().sum())

display(train.describe(include="all").T)

In [ ]:
target_summary = pd.DataFrame({
    "Count": train[TARGET].value_counts(),
    "Percent": (train[TARGET].value_counts(normalize=True) * 100).round(2)
})
display(target_summary)

train[TARGET].value_counts().plot(kind="bar")
plt.title("Target Distribution: Transported")
plt.xlabel("Transported")
plt.ylabel("Passengers")
plt.tight_layout()
plt.show()

In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": train.isna().sum(),
    "Missing_Percent": (train.isna().mean() * 100).round(2)
}).sort_values("Missing_Percent", ascending=False)

display(missing_summary)

missing_summary["Missing_Percent"].sort_values().plot(kind="barh", figsize=(8,6))
plt.title("Missing Values in Training Data")
plt.xlabel("Missing (%)")
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols_original = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

train[numeric_cols_original].hist(figsize=(13,8), bins=30)
plt.suptitle("Numerical Feature Distributions")
plt.tight_layout()
plt.show()

spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
eda_spend = train.copy()
eda_spend["TotalSpend"] = eda_spend[spend_cols].sum(axis=1, min_count=1)
eda_spend["NoSpend"] = (eda_spend[spend_cols].fillna(0).sum(axis=1) == 0)

display(
    eda_spend.groupby("NoSpend")[TARGET]
    .agg(["count", "mean"])
    .rename(columns={"mean":"Transport_Rate"})
)

In [ ]:
for col in ["HomePlanet", "CryoSleep", "Destination", "VIP"]:
    print(f"\nTransport rate by {col}")
    display(
        train.groupby(col, dropna=False)[TARGET]
        .agg(["count","mean"])
        .rename(columns={"mean":"Transport_Rate"})
        .sort_values("Transport_Rate", ascending=False)
    )

In [ ]:
temp = train.copy()

# Cabin components
cabin = (
    temp["Cabin"]
    .fillna("Unknown/Unknown/Unknown")
    .astype(str)
    .str.split("/", expand=True)
)
temp["CabinDeck"] = cabin[0]
temp["CabinSide"] = cabin[2]

print("Cabin Deck:")
display(
    temp.groupby("CabinDeck")[TARGET]
    .agg(["count","mean"])
    .rename(columns={"mean":"Transport_Rate"})
    .sort_values("Transport_Rate", ascending=False)
)

print("Cabin Side:")
display(
    temp.groupby("CabinSide")[TARGET]
    .agg(["count","mean"])
    .rename(columns={"mean":"Transport_Rate"})
)

# Passenger group
temp["GroupId"] = temp[ID].astype(str).str.split("_").str[0]
group_counts = temp["GroupId"].value_counts()
temp["GroupSize"] = temp["GroupId"].map(group_counts)
temp["TravelingSolo"] = temp["GroupSize"].eq(1)

print("Traveling Solo:")
display(
    temp.groupby("TravelingSolo")[TARGET]
    .agg(["count","mean"])
    .rename(columns={"mean":"Transport_Rate"})
)

In [ ]:
age_temp = train.copy()
age_temp["AgeGroup"] = pd.cut(
    age_temp["Age"],
    bins=[-np.inf, 12, 17, 25, 40, 60, np.inf],
    labels=["Child","Teen","YoungAdult","Adult","MiddleAge","Senior"]
)

age_result = (
    age_temp.groupby("AgeGroup", observed=False)[TARGET]
    .agg(["count","mean"])
    .rename(columns={"mean":"Transport_Rate"})
)
display(age_result)

age_result["Transport_Rate"].plot(kind="bar")
plt.title("Transport Rate by Age Group")
plt.ylabel("Transport Rate")
plt.tight_layout()
plt.show()